In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

## Simple Sequential Chain

In [2]:
load_dotenv()

True

In [3]:
prompt = PromptTemplate(
    template='write five point summary on {topic}',
    input_variables=['topic']
)

In [4]:
model = ChatGoogleGenerativeAI(model='gemini-3.5-flash-lite')

In [5]:
parser = StrOutputParser()

In [6]:
chain = prompt | model | parser

In [7]:
res = chain.invoke({'topic': 'Flutter framework'})

In [8]:
print(res)

Here is a five-point summary of the Flutter framework:

1. **Cross-Platform Development:** Flutter allows developers to build natively compiled applications for mobile (iOS and Android), web, desktop (Windows, macOS, Linux), and embedded devices from a single codebase.
2. **Dart Programming Language:** It uses Google’s object-oriented Dart language, which is easy to learn for developers familiar with languages like Java, C#, or JavaScript, and supports both Just-In-Time (JIT) and Ahead-Of-Time (AOT) compilation.
3. **Rich Widget Library:** Flutter is entirely widget-based; everything from layout structures to padding and buttons is a widget, enabling the creation of highly customizable, pixel-perfect UI designs that follow Material Design and Cupertino guidelines.
4. **High Performance:** Unlike many other cross-platform frameworks that rely on native bridges, Flutter bypasses them by using its own high-performance rendering engine (Impeller/Skia) to draw UI directly to the screen at a

In [9]:
prompt1 = PromptTemplate(
    template='create detail report on {topic}',
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template='create 5 points summary on {report}',
    input_variables=['report']
)

In [10]:
groq_model = ChatGroq(model='openai/gpt-oss-120b')

In [11]:
chain = prompt1 | model | parser | prompt2 | groq_model | parser

In [12]:
ch_res = chain.invoke({'topic': 'Nifty'})

In [13]:
print(ch_res)

**5‑Point Summary of the Nifty 50 Index (mid‑2024)**  

1. **Benchmark Profile & Methodology** – The Nifty 50, launched in 1996, is NSE’s flagship index tracking the 50 largest, most liquid Indian companies. It is calculated using a free‑float market‑cap weighted method and is re‑balanced semi‑annually (March & September).  

2. **Sector Composition** – The index is diversified but dominated by Financial Services (~33‑35% weight) and Information Technology (~13‑15%). Other notable sectors include Oil & Gas (~11‑12%), FMCG (~9‑10%) and Automobiles (~6‑7%).  

3. **Heavyweight Influence** – The top 10 constituents (e.g., Reliance Industries, HDFC Bank, ICICI Bank, Infosys, TCS, L&T, ITC, Bharti Airtel, Axis Bank, SBI) drive roughly 55‑60% of the index’s movement.  

4. **Historical Performance & Growth Drivers** – Over the past decade the Nifty 50 has delivered a CAGR of ~12‑15%, powered by the “financialisation” of Indian savings, record retail SIP inflows, steady domestic institutional

## Parallel and Conditional Chain

In [14]:
from langchain_core.runnables import RunnableParallel

In [15]:
pmt1 = PromptTemplate(
    template='Generate a summary on {topic}',
    input_variables=['topic']
)
pmt2 = PromptTemplate(
    template='Generate 5 quiz question on the {topic}',
     input_variables=['topic']
)

In [16]:
parser = StrOutputParser()

In [17]:
parallel_chain = RunnableParallel({'summary': pmt1 | model | parser, 'quiz': pmt2 | groq_model | parser})

In [18]:
pmt3 = PromptTemplate(
    template='merge the following {summary} and {quiz} together',
    input_variables=['summary', 'quiz']
)

In [19]:
chain3 = pmt3 | model | parser

In [20]:
final_chain = parallel_chain | chain3

In [21]:
ch_result = final_chain.invoke({'topic': 'Generative AI'})

In [22]:
print(ch_result)

Here is the merged text combining the comprehensive summary of **Generative Artificial Intelligence (GenAI)** and its accompanying **Knowledge Check Quiz** into a single, cohesive, and well-structured document. 

---

# Comprehensive Guide to Generative Artificial Intelligence (GenAI)

**Generative Artificial Intelligence (Generative AI or GenAI)** refers to a subset of artificial intelligence technologies capable of creating new content—including text, images, audio, video, code, and 3D data—in response to user prompts. 

Unlike traditional AI, which is designed to analyze data, classify information, or make predictions based on pre-existing rules, Generative AI learns the underlying patterns and structures of massive datasets to synthesize entirely original outputs.

Below is a complete overview of Generative AI, covering its core mechanisms, applications, impact, challenges, and a self-assessment quiz to test your understanding.

---

### 1. How It Works
Generative AI relies heavily

In [23]:
# !pip install grandalf

In [24]:
final_chain.get_graph().print_ascii()

             +-----------------------------+           
             | Parallel<summary,quiz>Input |           
             +-----------------------------+           
                    ***            ***                 
                  **                  **               
                **                      **             
    +----------------+             +----------------+  
    | PromptTemplate |             | PromptTemplate |  
    +----------------+             +----------------+  
             *                              *          
             *                              *          
             *                              *          
+------------------------+            +----------+     
| ChatGoogleGenerativeAI |            | ChatGroq |     
+------------------------+            +----------+     
             *                              *          
             *                              *          
             *                              *   

In [26]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.runnables import RunnableBranch, RunnableLambda
from typing import Literal, Optional
from pydantic import BaseModel, Field

In [27]:
class Sentiment(BaseModel):
    sentiment: Literal['Positive', 'Negative'] = Field(description='Generate the positive or negative as sentiment')

In [28]:
pyd_parser = PydanticOutputParser(pydantic_object=Sentiment)

In [38]:
prompt = PromptTemplate(
    template='Generate the sentiment of following {feedback} in {format}',
    input_variables=['feedback'],
    partial_variables={'format': pyd_parser.get_format_instructions()}
)

In [39]:
chain_s = prompt | model | pyd_parser

In [40]:
chain_s.invoke({'feedback': 'I did not liked this as its not working at all'}).sentiment

'Negative'

In [41]:
chain_s.invoke({'feedback': 'I liked this as its awesome'}).sentiment

'Positive'

In [42]:
str_parser = StrOutputParser()

In [43]:
prompt1 = PromptTemplate(
    template='Generate positive response for the received {sentiment} of the user',
    input_variables=['sentiment']
)

prompt2 = PromptTemplate(
    template='Generate response as sorry message for the received {sentiment} of the user',
    input_variables=['sentiment']
)

In [44]:
condn_chain = RunnableBranch(
    (lambda x: x.sentiment == 'Positive', prompt1 | model| str_parser),
    (lambda x: x.sentiment == 'Negative', prompt2 | model| str_parser),
    RunnableLambda(lambda x: 'Could not extract the sentiment')
)

In [45]:
final_chain = chain_s | condn_chain

In [47]:
res = final_chain.invoke('I phone duo is so expensive and very futuristic in the world of AI')

In [48]:
print(res)

Here are a few positive response options, depending on the context:

**Option 1: Warm & Enthusiastic (Best for customer service or general chat)**
> "That’s wonderful to hear! 😊 We're so glad you're feeling this way. How can we keep the good vibes going for you today?"

**Option 2: Short & Sweet**
> "That’s amazing! Thanks for sharing the positivity. ✨ Let me know if you need anything else!"

**Option 3: Appreciative (Best for feedback)**
> "Thank you so much for the kind words! It really brightened our day. Is there anything specific you'd like to dive into next?" 

**Option 4: Playful & Upbeat**
> "Love the energy! 🎉 Let's keep that momentum going. What's on your mind?"

*Feel free to let me know the specific context (e.g., customer support, gaming, wellness) if you want it tailored even further!*


In [49]:
res = final_chain.invoke('Realme phone are so bad as its lags a lot')
print(res)

I am truly sorry to hear that you are feeling this way or that your experience wasn't what it should have been. 

We always strive to do better, and I would love the chance to make things right. Please let me know what went wrong or how I can help you turn this around. 

Thank you for your patience and for letting us know.


In [50]:
final_chain.get_graph().print_ascii()

      +-------------+      
      | PromptInput |      
      +-------------+      
             *             
             *             
             *             
    +----------------+     
    | PromptTemplate |     
    +----------------+     
             *             
             *             
             *             
+------------------------+ 
| ChatGoogleGenerativeAI | 
+------------------------+ 
             *             
             *             
             *             
 +----------------------+  
 | PydanticOutputParser |  
 +----------------------+  
             *             
             *             
             *             
        +--------+         
        | Branch |         
        +--------+         
             *             
             *             
             *             
     +--------------+      
     | BranchOutput |      
     +--------------+      
